In [ ]:
#%% md
# Ноутбук 5: Релевантность/новизна события (REL) + отраслевой spillover-эффект
#
# Гипотеза I (событийная релевантность, REL): не все упоминания тикера в тексте одинаково
#     важны — упоминание в начале новости и многократно значит больше, чем случайное
#     упоминание в конце длинного текста. Строим непрерывный REL-скор (0-100) по позиции
#     первого упоминания и количеству упоминаний тикера (аналог RavenPack REL).
#
# Гипотеза J (attention spillover): новости об одном тикере двигают цены СВЯЗАННЫХ бумаг
#     того же сектора, не только самого упомянутого тикера. Проверяем регрессией:
#     |доходность| тикера ~ новости про сам тикер + новости про его сектор-соседей.
#
# Обе гипотезы совмещены в одном ноутбуке, потому что вторая использует первую как фильтр
# (ablation: регрессия на "всех" упоминаниях тикера vs только достаточно "релевантных"
# REL >= порога) — это одна логическая цепочка: сначала уточняем сигнал (I), потом проверяем
# его способность объяснить движения СВЯЗАННЫХ бумаг (J).
#
# НЕ включаем сюда дедупликацию как таковую (базовый ноутбук), сравнение эмбеддингов
# (ноутбук 2) и новостную новизну по времени (ноутбук 1) — здесь фокус на связке
# тикер -> сектор -> рынок.
#%%


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DATA_PATH = '/kaggle/input/datasets/artemwalker/telegram-news/telegram_news_raw.csv'
OUT_DIR = Path('relevance_spillover_results')
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=['post_dttm']).reset_index(drop=True)

def normalize_text(text):
    text = str(text or '').lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[\w_]+', ' ', text)
    text = re.sub(r'[^\w\s%.-]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

df['text_normalized'] = df['data'].fillna('').map(normalize_text)
df = df[df.text_normalized.str.len().gt(0)].copy().reset_index(drop=True)
print(f'Новостей: {len(df):,}')

In [ ]:
FALLBACK_SECTOR_MAP = {
    'SBER': 'Финансы', 'VTBR': 'Финансы',
    'GAZP': 'Нефть и газ', 'LKOH': 'Нефть и газ', 'ROSN': 'Нефть и газ',
    'NVTK': 'Нефть и газ', 'TATN': 'Нефть и газ', 'SNGS': 'Нефть и газ',
    'GMKN': 'Металлургия', 'CHMF': 'Металлургия', 'ALRS': 'Металлургия',
    'RUAL': 'Металлургия', 'PLZL': 'Металлургия', 'POLY': 'Металлургия',
    'MTSS': 'Телеком', 'MGNT': 'Ритейл', 'YDEX': 'Технологии',
    'MOEX': 'Финансы', 'AFLT': 'Транспорт', 'PHOR': 'Химия',
}

def load_moex_tickers():
    """Полный список тикеров MOEX через ISS API; поле сектора там обычно не отдаётся,
    поэтому для группировки по секторам используем запасную ручную карту по крупнейшим
    бумагам (FALLBACK_SECTOR_MAP), а остальные относим к сектору 'Прочее'."""
    url = 'https://iss.moex.com/iss/engines/stock/markets/shares/securities.json'
    try:
        resp = requests.get(url, params={'iss.meta': 'off'}, timeout=20)
        resp.raise_for_status()
        payload = resp.json()['securities']
        secid_i = payload['columns'].index('SECID')
        tickers = sorted({r[secid_i] for r in payload['data'] if r[secid_i]})
        if tickers:
            return tickers
        raise ValueError('пустой список')
    except Exception as exc:
        print(f'MOEX ISS недоступен ({exc}), использую запасной список.')
        return list(FALLBACK_SECTOR_MAP.keys())

MOEX_TICKERS = load_moex_tickers()
sector_map = {t: FALLBACK_SECTOR_MAP.get(t, 'Прочее') for t in MOEX_TICKERS}

ticker_pattern = re.compile(r'(?:\$)?\b(' + '|'.join(re.escape(t) for t in MOEX_TICKERS) + r')\b')

def find_ticker_positions(text):
    text = str(text or '')
    return [(m.group(1), m.start()) for m in ticker_pattern.finditer(text)]

df['ticker_hits'] = df['data'].map(find_ticker_positions)
print(f'Новостей с упоминанием тикера: {(df.ticker_hits.map(len) > 0).sum():,}')

In [ ]:
#%% md
### Гипотеза I: событийная релевантность (REL-скор)
# Как применяем: для каждой пары (новость, тикер) считаем:
#   - позицию первого упоминания (чем раньше в тексте — тем релевантнее);
#   - число упоминаний в тексте (чем чаще — тем релевантнее, с насыщением).
# REL = 100 * (0.6 * position_score + 0.4 * count_score).
# Как оцениваем результат: смотрим распределение REL по всем парам новость-тикер и долю
#     пар с высоким REL (>= REL_THRESHOLD) — именно они пойдут в "отфильтрованную" регрессию
#     гипотезы J.
#%%
def compute_rel_scores(row):
    text_len = max(len(str(row['data'] or '')), 1)
    hits = row['ticker_hits']
    by_ticker = {}
    for ticker, pos in hits:
        by_ticker.setdefault(ticker, []).append(pos)
    scores = {}
    for ticker, positions in by_ticker.items():
        first_pos = min(positions)
        position_score = max(0.0, 1 - first_pos / text_len)
        count_score = 1 - 1 / (1 + len(positions))  # насыщающийся рост с числом упоминаний
        scores[ticker] = round(100 * (0.6 * position_score + 0.4 * count_score), 1)
    return scores

df['ticker_rel_scores'] = df.apply(compute_rel_scores, axis=1)

rel_rows = []
for idx, row in df.iterrows():
    for ticker, rel in row['ticker_rel_scores'].items():
        rel_rows.append({'row_id': idx, 'ticker': ticker, 'post_dttm': row.post_dttm, 'rel': rel})
rel_df = pd.DataFrame(rel_rows)
rel_df['sector'] = rel_df['ticker'].map(sector_map)

fig, ax = plt.subplots(figsize=(9, 4))
rel_df['rel'].hist(bins=30, ax=ax)
ax.set_title('Распределение REL-скора по парам (новость, тикер)')
ax.set_xlabel('REL (0-100)')
ax.set_ylabel('Число пар')
plt.tight_layout()
plt.savefig(OUT_DIR / 'rel_score_hist.png', dpi=150)
plt.show()

REL_THRESHOLD = 70
print(f'Пар с REL >= {REL_THRESHOLD}: {(rel_df.rel >= REL_THRESHOLD).mean():.1%}')
rel_df.to_csv(OUT_DIR / 'rel_scores.csv', index=False, encoding='utf-8-sig')

In [ ]:
#%% md
### Гипотеза J: отраслевой spillover-эффект
# Как применяем: для тикеров с доступными котировками MOEX считаем на каждый торговый день:
#   own_news_count — число новостей с упоминанием САМОГО тикера,
#   peer_news_count — число новостей с упоминанием ДРУГИХ тикеров того же сектора.
# Строим OLS: |return| ~ own_news_count + peer_news_count, отдельно на "всех" упоминаниях
#     (без фильтра REL) и только на REL >= REL_THRESHOLD — чтобы увидеть, помогает ли
#     фильтрация по релевантности выделить более сильный и значимый spillover-эффект.
# Как оцениваем результат: сравниваем R² и коэффициент/p-value при peer_news_count между
#     двумя версиями регрессии (все упоминания vs только релевантные) в одной сводной таблице.
#%%
try:
    import statsmodels.api as sm
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    import statsmodels.api as sm

def load_moex_security_history(secid, board='TQBR', start=None, end=None):
    base = f'https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/{board}/securities/{secid}.json'
    frames, start_pos = [], 0
    while True:
        params = {'iss.meta': 'off', 'start': start_pos}
        if start: params['from'] = start
        if end: params['till'] = end
        try:
            resp = requests.get(base, params=params, timeout=20)
            resp.raise_for_status()
        except Exception:
            return pd.DataFrame()
        payload = resp.json()['history']
        rows = payload['data']
        if not rows:
            break
        frames.append(pd.DataFrame(rows, columns=payload['columns']))
        start_pos += len(rows)
        if len(rows) < 100:
            break
    if not frames:
        return pd.DataFrame()
    hist = pd.concat(frames, ignore_index=True)
    hist['TRADEDATE'] = pd.to_datetime(hist['TRADEDATE'])
    return hist[['TRADEDATE', 'CLOSE']].dropna().drop_duplicates('TRADEDATE').sort_values('TRADEDATE')

TOP_N_TICKERS = 12
top_tickers = rel_df['ticker'].value_counts().head(TOP_N_TICKERS).index.tolist()
market_start = df.post_dttm.min().date().isoformat()
market_end = df.post_dttm.max().date().isoformat()

price_returns = {}
for t in tqdm(top_tickers, desc='Загрузка котировок'):
    hist = load_moex_security_history(t, start=market_start, end=market_end)
    if not hist.empty:
        s = hist.set_index('TRADEDATE')['CLOSE'].pct_change().abs()
        price_returns[t] = s

print(f'Получены котировки для {len(price_returns)} из {len(top_tickers)} тикеров.')

def build_regression_dataset(rel_min):
    subset = rel_df[rel_df.rel >= rel_min].assign(day=lambda x: x.post_dttm.dt.date)
    rows = []
    for ticker in price_returns:
        sector = sector_map.get(ticker, 'Прочее')
        own = subset[subset.ticker == ticker].groupby('day').size().rename('own_news_count')
        peers = subset[(subset.sector == sector) & (subset.ticker != ticker)].groupby('day').size().rename('peer_news_count')
        ret = price_returns[ticker].rename('abs_return')
        ret.index = ret.index.date
        merged = pd.concat([ret, own, peers], axis=1).fillna(0).dropna(subset=['abs_return'])
        merged['ticker'] = ticker
        rows.append(merged)
    return pd.concat(rows) if rows else pd.DataFrame()

def run_spillover_regression(rel_min, label):
    data = build_regression_dataset(rel_min)
    if len(data) < 10:
        print(f'{label}: недостаточно данных ({len(data)} наблюдений) — пропускаем.')
        return None
    X = sm.add_constant(data[['own_news_count', 'peer_news_count']])
    y = data['abs_return']
    model = sm.OLS(y, X).fit()
    print(f'--- {label}: n={int(model.nobs)}, R^2={model.rsquared:.4f} ---')
    print(model.summary().tables[1])
    return model

model_all = run_spillover_regression(0, 'Все упоминания (без фильтра REL)')
model_filtered = run_spillover_regression(REL_THRESHOLD, f'Только REL >= {REL_THRESHOLD}')

def get_param(model, name, table):
    if model is None:
        return np.nan
    return getattr(model, table)[name] if name in getattr(model, table) else np.nan

comparison = pd.DataFrame({
    'model': ['все упоминания', f'REL >= {REL_THRESHOLD}'],
    'r_squared': [getattr(model_all, 'rsquared', np.nan), getattr(model_filtered, 'rsquared', np.nan)],
    'peer_coef': [get_param(model_all, 'peer_news_count', 'params'),
                  get_param(model_filtered, 'peer_news_count', 'params')],
    'peer_pvalue': [get_param(model_all, 'peer_news_count', 'pvalues'),
                    get_param(model_filtered, 'peer_news_count', 'pvalues')],
})
print(comparison)
comparison.to_csv(OUT_DIR / 'spillover_regression_comparison.csv', index=False, encoding='utf-8-sig')

In [ ]:
# Итог ноутбука 5:
#  - rel_scores.csv — REL-скор релевантности по всем парам новость-тикер (гипотеза I)
#  - spillover_regression_comparison.csv — сравнение регрессии с/без фильтра по REL (гипотеза J)
print('Готово. Файлы результатов сохранены в', OUT_DIR.resolve())